# Download English Wikipedia and select Parquet shards

**Dataset:** [Wikipedia — wikimedia/wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia).

This notebook downloads **only the English snapshot `20231101.en`**, pinned to revision `b04c8d1ceb2f5cd4588862100d08de323dccfbaa`. Its 41 compressed Parquet shards total **11,630,929,031 bytes: 11.63 GB (10.83 GiB), commonly described as approximately 11 GB**. This is not a download of every language or the latest live Wikipedia.

It replaces the former full-RAG notebook with **data acquisition and shard selection only**. It does not load a language model, generate embeddings, construct SQLite/PostgreSQL indexes or answer questions. The working PGVector RAG notebook and its database are unaffected.

**First run:** create the data directory and automatically download missing selected shards. **Subsequent runs:** validate existing files and skip valid ones. Partial folders are completed rather than treated as fully downloaded. Internet is needed only for missing/invalid files or the download dependency.

Run cells from top to bottom. The default downloads all 41 English shards. The separate RAG slice selects shards 0–7 without copying their contents.

## Portable project folder and database settings
Keep notebooks 01–05 together in the project folder. All data, models and results use paths relative to that folder through `pathlib`, on macOS, Windows or Linux. The code searches the kernel working directory and its parents for notebook 01. Jupyter does not reliably expose the open notebook's file path; if its working directory is elsewhere, set `RAG_PROJECT_DIR` to the project folder before running configuration. A relative override is resolved from the kernel working directory.

Optional override in a cell before configuration:
```python
import os
os.environ['RAG_PROJECT_DIR'] = 'path/to/your/project'  # Relative, or your native absolute path.
```

Database settings in 03–05 follow this precedence: `RAG_PG_DSN` / `RAG_PG_SCHEMA` environment overrides, then `data/rag_setup_connection.json` saved by 03, then the local defaults. On a new machine, run 03 to prepare the connection before 04/05. Supply the PostgreSQL role in the DSN when it differs from your operating-system username. Credentials remain in your environment or PostgreSQL password file; do not put passwords in notebook cells. A saved endpoint is machine-specific: override it when moving to another server. Notebook 03 saves the effective endpoint without its password.

The scope is intentional: notebook 01 downloads **all 41 English shards**, and notebook 04 embeds only **00000–00007** to limit computation. No download or embedding scope is changed by the portable paths.


## 1. Configuration

Default destination: `IITB-DL/data/wikipedia_snapshot/20231101.en/`.

- Keep `DOWNLOAD_SHARDS=None` for the full English dataset (~11 GB).
- Set `DOWNLOAD_SHARDS=list(range(8))` to acquire only shards 0–7 (a smaller download, not the full 11 GB).
- `RAG_SLICE=list(range(8))` records which existing shards the current RAG uses; it does not control or extend database ingestion.
- Default validation checks exact file sizes. Set `VERIFY_SHA256=True` to also detect same-size corruption; hashing all 11 GB takes longer.
- No files outside the selection are deleted. Changing selection does not remove previously downloaded shards.

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys

# Paths are relative to the project folder, using native macOS/Windows separators.
def resolve_project_root():
    override = os.environ.get('RAG_PROJECT_DIR')
    if override:
        folder = Path(override).expanduser().resolve()
        if not folder.is_dir():
            raise FileNotFoundError('RAG_PROJECT_DIR must point to an existing project folder.')
        return folder
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder/'01_download_english_wikipedia_dataset.ipynb').is_file():
            return folder
    raise RuntimeError('Open Jupyter from the folder containing notebooks 01–05, or set RAG_PROJECT_DIR to that folder.')

ROOT = resolve_project_root()
PROJECT_DIR = ROOT
DATASET_DIR = PROJECT_DIR / 'data' / 'wikipedia_snapshot'
DATASET = 'wikimedia/wikipedia'
REVISION = 'b04c8d1ceb2f5cd4588862100d08de323dccfbaa'
CONFIG = '20231101.en'
DOWNLOAD_SHARDS = None  # None = all 41 English shards; list(range(8)) = only shards 0–7.
RAG_SLICE = list(range(8))  # Record the existing RAG's first-eight-shard selection; no copying.
ALLOW_DOWNLOADS = True  # First run automatically downloads missing files; complete files are reused.
VERIFY_SHA256 = False  # True reads all selected files for stronger, slower integrity verification.
DOWNLOAD_WORKERS = 4
INSTALL_MISSING_DEPENDENCY = True  # huggingface_hub only, and only if a download is required.


## 2. Pinned English-only file inventory

Each entry specifies the upstream shard path, expected compressed size and SHA-256. An embedded inventory lets a complete local dataset be reused without contacting Hugging Face. These files are already sharded by the dataset publisher: we preserve their names and contents instead of downloading one large file and splitting it again.

In [2]:
ENGLISH_FILES = [{'path': '20231101.en/train-00000-of-00041.parquet', 'size': 420296449, 'sha256': '382e7f6f09e488b24793a7f7cfc659879d5a22da2cf2efec6491665f0c019677'}, {'path': '20231101.en/train-00001-of-00041.parquet', 'size': 351448367, 'sha256': '6cd57a5825312ca966313d4a1b614690b5c4fb53cee9b124bc0e999977eb36cf'}, {'path': '20231101.en/train-00002-of-00041.parquet', 'size': 329024669, 'sha256': 'b18674f163d4f106e43396fd58412a79b80a418c72bb728c20af439e7ca4d5b7'}, {'path': '20231101.en/train-00003-of-00041.parquet', 'size': 330779424, 'sha256': '0b8a6f452f5ef59d0986a067f70110feb16d7c1d281ba2db0b3df32933618fcc'}, {'path': '20231101.en/train-00004-of-00041.parquet', 'size': 306945391, 'sha256': '1424300f113c13ee022b7929185fa4f7e20c6195ba563d728e8a3dc47c8e9aba'}, {'path': '20231101.en/train-00005-of-00041.parquet', 'size': 243974056, 'sha256': '220d31bb222ba2d449c845733622a6a2b7a89db3e28b24bfcb3a9bb3e52c6c8d'}, {'path': '20231101.en/train-00006-of-00041.parquet', 'size': 266288728, 'sha256': 'b8becbe9989a2c63f129a707a4674a84fb093c163673b1f4ce97afbf5306f04c'}, {'path': '20231101.en/train-00007-of-00041.parquet', 'size': 228195804, 'sha256': 'adebba8b075444923ba42ed6214fe205efc756c773f440d98cbdd88a33597c2c'}, {'path': '20231101.en/train-00008-of-00041.parquet', 'size': 247687191, 'sha256': '905f959921ab8de07e7ce238d7c9ef33bd597585a17a99c3ba00163b25002bef'}, {'path': '20231101.en/train-00009-of-00041.parquet', 'size': 227378231, 'sha256': '465bd99719b0f2d1974402b7922a7a66399675dc6218a0a40057e0512592c07b'}, {'path': '20231101.en/train-00010-of-00041.parquet', 'size': 233604795, 'sha256': 'cd8b4e9ee2506ba5f47d552c56e2a7569402a6814d9fa171c0e995e56859e51b'}, {'path': '20231101.en/train-00011-of-00041.parquet', 'size': 231748583, 'sha256': '881e8efe6dbaddd34791fdc2158123ed344c436263b9052a0d659418fff89c3a'}, {'path': '20231101.en/train-00012-of-00041.parquet', 'size': 239222511, 'sha256': 'd33835c5e69d5f2464cab1ad70b998ec37e084ea0c0ef9f837fac4a9bb030860'}, {'path': '20231101.en/train-00013-of-00041.parquet', 'size': 241146961, 'sha256': '9abcab8c808952dc6a4822da1591db51de980a792ed37acd3bd243ebd99d200e'}, {'path': '20231101.en/train-00014-of-00041.parquet', 'size': 222798405, 'sha256': '7419912e150b17ce395ea039620c21a7e1cb54731c7c2fd2ccbbed56674eeccd'}, {'path': '20231101.en/train-00015-of-00041.parquet', 'size': 234578705, 'sha256': '1d11ff44d5579dbb9e1976330cdf70ec8b04a7a5dfe4113916b71551fb3cf715'}, {'path': '20231101.en/train-00016-of-00041.parquet', 'size': 503153575, 'sha256': 'e559da89b795e365bc1d1bb87576d1a5478c346cf37eb06159caf9579fe1e54e'}, {'path': '20231101.en/train-00017-of-00041.parquet', 'size': 230813705, 'sha256': '216cd1e4921cc884ebdaed94b61d3e15560f502f73752f3694ed05e6347a6562'}, {'path': '20231101.en/train-00018-of-00041.parquet', 'size': 231055386, 'sha256': '64fb53939ab6aae24299cdafa2ce88e115d91930be35609b2dcfc1fc3b9f85eb'}, {'path': '20231101.en/train-00019-of-00041.parquet', 'size': 195276969, 'sha256': '99c23125b0dd115d68f4a921376880e307bdfcbf3bfd66ae4ed90e853e1dadac'}, {'path': '20231101.en/train-00020-of-00041.parquet', 'size': 225141733, 'sha256': '79ee06627f4ba2fec056fcd23c077a29238bbcbacc3187e8d1a0b47e663e9c7a'}, {'path': '20231101.en/train-00021-of-00041.parquet', 'size': 216445587, 'sha256': 'e9f695672d916793439d8f48893ecde94c61b0eb038a0034c4e1f1f194205979'}, {'path': '20231101.en/train-00022-of-00041.parquet', 'size': 201995404, 'sha256': '50ba80435bf2fd0b818533a1ca3118454a3ae275dc2250e0b68dc852dd03f3d2'}, {'path': '20231101.en/train-00023-of-00041.parquet', 'size': 212615391, 'sha256': 'b722b7e4039c65382286b6aa5a35f9a4c5d3536b758c346a94af7a2a692cc6bb'}, {'path': '20231101.en/train-00024-of-00041.parquet', 'size': 220729597, 'sha256': 'eb9818bff41110829f39f8100aee5936cf6088c72a5925c2b08bcd3c37ac0170'}, {'path': '20231101.en/train-00025-of-00041.parquet', 'size': 221292721, 'sha256': 'fed7d34add9b161745dd2677ff3c5f782a471f433bae80c59933e9b68225be93'}, {'path': '20231101.en/train-00026-of-00041.parquet', 'size': 208188233, 'sha256': 'ffffb7d303ef78b47c4c835211b35f6052ffa2d5998e3f65d69bb7fe512ee69c'}, {'path': '20231101.en/train-00027-of-00041.parquet', 'size': 213980716, 'sha256': '1f0c7bcaf2aeb9765a981c8911b95ab3a9e50456bbad64e50e722a53b8608ccc'}, {'path': '20231101.en/train-00028-of-00041.parquet', 'size': 188376392, 'sha256': '10589a39188af404fa458da252df7dcf22c6f6395a1501d9243ef56ce4c3c148'}, {'path': '20231101.en/train-00029-of-00041.parquet', 'size': 217775627, 'sha256': '8f24c4250d69dbee36962d587cac632098635fa78e8705db140cce68928bb9d3'}, {'path': '20231101.en/train-00030-of-00041.parquet', 'size': 203945186, 'sha256': '1de329963a4aaed0c40d820a564487dc98ef113c8ca6b88b11d495d6a4b98774'}, {'path': '20231101.en/train-00031-of-00041.parquet', 'size': 215470925, 'sha256': '55e58e33710b9326e1ce3aab1319377afd9ce18df089b08aba2ab9286228666e'}, {'path': '20231101.en/train-00032-of-00041.parquet', 'size': 214242576, 'sha256': '354c4d35fa089351ed4ec1622fd8930f309dddd3dccbf411e4dd8a52eff638d4'}, {'path': '20231101.en/train-00033-of-00041.parquet', 'size': 202706223, 'sha256': 'a5140a1c80805d599b791fbd668d0f49972eecf0cd2d863cede53013f52e0900'}, {'path': '20231101.en/train-00034-of-00041.parquet', 'size': 219010799, 'sha256': '298528554e93a3fae9f694087a7376b70be223084b2739ef9488418202a948ab'}, {'path': '20231101.en/train-00035-of-00041.parquet', 'size': 224459876, 'sha256': '460ab2f5a2da091e310d0f86c461a73f22b189f22ac7115e4708a5da9baf4d1c'}, {'path': '20231101.en/train-00036-of-00041.parquet', 'size': 610276598, 'sha256': '7385c9f0482cd592578c1c22c5eaaad8bc3013bec4e3bfe394b441dc728f78a9'}, {'path': '20231101.en/train-00037-of-00041.parquet', 'size': 674143674, 'sha256': 'f2f7718efc3165ed599ec0d99fef07b98ec4dd0da902ffc330dd5c75016240cb'}, {'path': '20231101.en/train-00038-of-00041.parquet', 'size': 538231796, 'sha256': '16ff05ab829cb9c983db14b7f77a677d279f8120571e67b2793735851b71ae89'}, {'path': '20231101.en/train-00039-of-00041.parquet', 'size': 464851854, 'sha256': '63a0be9b1b458b834428c897e37a6e2643e0c33dd4ac08a994f3b3dbeee124e1'}, {'path': '20231101.en/train-00040-of-00041.parquet', 'size': 421630218, 'sha256': 'fed6addb1da9364fcf42c6953539eecc21bbc2cf3cd33743684e6872ce6c468a'}]

## 3. Download and validation helpers

The downloader requests only the selected English paths at the pinned revision. Missing files and incomplete/invalid files are handled separately; valid files are skipped. Rerunning retries interrupted work using Hugging Face's local download cache. A failed or interrupted download is not reported as complete.

A fresh subprocess clears inherited Hugging Face offline flags for the download only. The parent notebook's environment is not changed. Only `huggingface-hub` is needed for downloading; PyTorch, Transformers and database packages are not required.

In [3]:
def _download_files(destination: Path, spec: dict, entries: list[dict], max_workers: int):
    # A fresh process can download even if an earlier notebook cell imported Hub
    # with HF_HUB_OFFLINE=1. The parent kernel's offline settings remain unchanged.
    payload = {
        "repo_id": spec["repo_id"], "repo_type": spec["repo_type"], "revision": spec["revision"],
        "local_dir": str(destination.resolve()), "max_workers": max_workers,
        "missing": [e["path"] for e in entries if not (destination / e["path"]).exists()],
        "repair": [e["path"] for e in entries if (destination / e["path"]).exists()],
    }
    program = '''
import json, sys
from huggingface_hub import snapshot_download
p = json.load(sys.stdin)
for group, force in (("missing", False), ("repair", True)):
    if p[group]:
        snapshot_download(
            repo_id=p["repo_id"], repo_type=p["repo_type"], revision=p["revision"],
            local_dir=p["local_dir"], allow_patterns=p[group],
            force_download=force, max_workers=p["max_workers"],
        )
'''
    environment = os.environ.copy()
    for name in ("HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE", "TRANSFORMERS_OFFLINE"):
        environment[name] = "0"
    environment["HF_HUB_DISABLE_TELEMETRY"] = "1"
    environment["HF_XET_CACHE"] = str(destination / ".cache/huggingface/xet")
    with subprocess.Popen(
        [sys.executable, "-u", "-c", program], stdin=subprocess.PIPE,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=environment,
    ) as process:
        try:
            process.stdin.write(json.dumps(payload))
            process.stdin.close()
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait() != 0:
                raise RuntimeError("Dataset download interrupted or failed. Rerun to resume.")
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise

def selected_entries(indices):
    if indices is None: indices = list(range(41))
    if not indices or any(type(i) is not int or not 0 <= i < 41 for i in indices):
        raise ValueError('Select one or more integer shard indices from 0 through 40.')
    if len(set(indices)) != len(indices): raise ValueError('Duplicate shard indices are not allowed.')
    return [ENGLISH_FILES[i] for i in sorted(indices)]

def validate_inventory():
    expected = [f'{CONFIG}/train-{i:05d}-of-00041.parquet' for i in range(41)]
    if [e['path'] for e in ENGLISH_FILES] != expected:
        raise ValueError('Inventory must contain exactly the 41 pinned English Parquet paths.')
    if sum(e['size'] for e in ENGLISH_FILES) != 11630929031:
        raise ValueError('Unexpected pinned English dataset size.')
    if any(len(e['sha256']) != 64 for e in ENGLISH_FILES): raise ValueError('Missing SHA-256.')

def file_valid(path,entry,verify_hashes=False):
    if not path.is_file() or path.stat().st_size != entry['size']: return False
    if not verify_hashes: return True
    digest=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):digest.update(block)
    return digest.hexdigest()==entry['sha256']

def inspect_files(destination,entries,verify_hashes=False):
    invalid=[e for e in entries if not file_valid(destination/e['path'],e,verify_hashes)]
    return dict(total=len(entries),valid=len(entries)-len(invalid),missing_or_invalid=invalid,
                bytes_to_download=sum(e['size'] for e in invalid),complete=not invalid)

def ensure_download_dependency():
    if importlib.util.find_spec('huggingface_hub') is None:
        if not INSTALL_MISSING_DEPENDENCY:
            raise RuntimeError('Install huggingface-hub in this notebook kernel to download missing files.')
        subprocess.check_call([sys.executable,'-m','pip','install','huggingface-hub==0.36.2'])

def ensure_dataset(destination,entries,*,allow_download=True,verify_hashes=False,max_workers=4):
    validate_inventory()
    allowed={e['path']:e for e in ENGLISH_FILES}
    if not entries or any(allowed.get(e['path'])!=e for e in entries):
        raise ValueError('Only pinned English shard entries may be downloaded.')
    if type(max_workers) is not int or max_workers<1:raise ValueError('max_workers must be positive.')
    destination=Path(destination)
    status=inspect_files(destination,entries,verify_hashes)
    print(f"Selected shards: {status['total']}; valid local files: {status['valid']}",flush=True)
    if status['complete']:
        print('Download skipped: all selected files are already present. No network or package installation needed.')
        return status
    if not allow_download:
        raise FileNotFoundError(f"{len(status['missing_or_invalid'])} shards are missing/invalid. Enable ALLOW_DOWNLOADS while online.")
    destination.mkdir(parents=True,exist_ok=True)
    required=status['bytes_to_download']+10**9  # Conservative space estimate plus 1 GB working reserve.
    free=shutil.disk_usage(destination).free
    if free<required:
        raise OSError(f'Need approximately {required/1e9:.2f} GB free including reserve; {free/1e9:.2f} GB available.')
    ensure_download_dependency()
    print(f"Downloading/resuming {len(status['missing_or_invalid'])} shards ({status['bytes_to_download']/1e9:.2f} GB maximum); progress follows.",flush=True)
    _download_files(destination,dict(repo_id=DATASET,repo_type='dataset',revision=REVISION),status['missing_or_invalid'],max_workers)
    status=inspect_files(destination,entries,verify_hashes)
    if not status['complete']:raise RuntimeError('Download incomplete. Rerun to retry missing/invalid files.')
    print('All selected English Parquet shards are now available locally.')
    return status

def write_selection_manifest(path,entries,verify_hashes=False):
    # A manifest selects existing shards; it does not duplicate, rechunk, or index data.
    status=inspect_files(DATASET_DIR,entries,verify_hashes)
    value=dict(dataset=DATASET,revision=REVISION,language='en',snapshot=CONFIG,
        complete=status['complete'],validation='sha256' if verify_hashes else 'file_size',
        compressed_bytes=sum(e['size'] for e in entries),
        files=[dict(e,local_path=str((DATASET_DIR/e['path']).resolve())) for e in entries])
    path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix('.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');temp.replace(path)
    return value


## 4. Check local files and download only what is needed

This is the acquisition step. It prints the selected size, local completeness and download progress. It does not contact the network when the selected files already pass validation.

In [4]:
validate_inventory()
entries=selected_entries(DOWNLOAD_SHARDS)
print('Dataset:',DATASET,'| English snapshot:',CONFIG)
print(f'Selected compressed size: {sum(e["size"] for e in entries)/1e9:.2f} GB')
print('Destination:',DATASET_DIR/CONFIG)
status=ensure_dataset(DATASET_DIR,entries,allow_download=ALLOW_DOWNLOADS,
                      verify_hashes=VERIFY_SHA256,max_workers=DOWNLOAD_WORKERS)


Dataset: wikimedia/wikipedia | English snapshot: 20231101.en
Selected compressed size: 11.63 GB
Destination: /Users/amolsingh/Documents/code/IITB-DL/data/wikipedia_snapshot/20231101.en
Selected shards: 41; valid local files: 41
Download skipped: all selected files are already present. No network or package installation needed.


## 5. Slice by shard and save manifests

“Slicing” here means selecting whole upstream Parquet shards. For example, the current RAG uses `train-00000-of-00041.parquet` through `train-00007-of-00041.parquet`. Shards are not subject/domain categories and are not equal in size. This step does not split articles into RAG chunks; that belongs to ingestion.

Two small JSON manifests record the downloaded selection and the intended RAG slice, with source revision, paths and sizes. If the requested RAG slice is not present, its manifest explicitly says `complete: false`. The manifest does not download extra files or modify the RAG database.

In [5]:
download_manifest=write_selection_manifest(DATASET_DIR/'english_download_selection.json',entries,VERIFY_SHA256)
slice_entries=selected_entries(RAG_SLICE)
slice_manifest=write_selection_manifest(DATASET_DIR/'english_rag_slice.json',slice_entries,VERIFY_SHA256)
print('Download selection complete:',download_manifest['complete'])
print(f"RAG slice: {len(slice_entries)} shards, {slice_manifest['compressed_bytes']/1e9:.2f} GB; locally complete: {slice_manifest['complete']}")
for e in slice_entries:
    print(e['path'],f"{e['size']/1e6:.1f} MB")
print('Manifests saved in:',DATASET_DIR)


Download selection complete: True
RAG slice: 8 shards, 2.48 GB; locally complete: True
20231101.en/train-00000-of-00041.parquet 420.3 MB
20231101.en/train-00001-of-00041.parquet 351.4 MB
20231101.en/train-00002-of-00041.parquet 329.0 MB
20231101.en/train-00003-of-00041.parquet 330.8 MB
20231101.en/train-00004-of-00041.parquet 306.9 MB
20231101.en/train-00005-of-00041.parquet 244.0 MB
20231101.en/train-00006-of-00041.parquet 266.3 MB
20231101.en/train-00007-of-00041.parquet 228.2 MB
Manifests saved in: /Users/amolsingh/Documents/code/IITB-DL/data/wikipedia_snapshot


## Completion and hand-off

The Parquet files remain at `data/wikipedia_snapshot/20231101.en/`. The first-eight-shard manifest is a reproducible file selection, not an embedding or index-completion record. Use `05_offline_wikipedia_hybrid_rag.ipynb` for the existing retrieval/generation pipeline; this download notebook never builds or changes its indexes.

If downloads fail, check internet access, permissions and free space, then rerun. Do not delete completed shards to resume. On another machine, open Jupyter from the project folder or set `RAG_PROJECT_DIR`; paths resolve automatically. If the upstream files become unavailable at the pinned revision, investigate explicitly rather than silently mixing a different snapshot into the same directory.